# WC 2022 — Schema Exploration
Loads the three parquet files and documents dtypes, shapes, and key nested structures.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path
import json

CACHE = Path('../data/cache')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 120)

## 1. matches.parquet

In [ ]:
matches = pd.read_parquet(CACHE / 'matches.parquet')
print('Shape:', matches.shape)
print()
print('dtypes:')
print(matches.dtypes.to_string())

In [ ]:
matches.head(3)

## 2. events.parquet

In [ ]:
events = pd.read_parquet(CACHE / 'events.parquet')
print('Shape:', events.shape)
print()
print('dtypes:')
print(events.dtypes.to_string())

In [ ]:
events.head(3)

In [ ]:
print('Event type counts:')
print(events['type'].value_counts().to_string())

In [ ]:
for etype in ['Shot', 'Pass', 'Carry', 'Pressure']:
    subset = events[events['type'] == etype]
    print(f'\n=== {etype} (n={len(subset)}) — sample row ===')
    row = subset.iloc[0]
    print(row.dropna().to_string())

In [ ]:
# Shot nested dict fields
shot_cols = [c for c in events.columns if c.startswith('shot')]
print('Shot-related columns:', shot_cols)
print()
shots = events[events['type'] == 'Shot'].copy()
print(f'Total shots: {len(shots)}')
print()
for col in shot_cols:
    sample = shots[col].dropna()
    if len(sample) > 0:
        print(f'--- {col} ---')
        val = sample.iloc[0]
        if isinstance(val, dict):
            print(json.dumps(val, indent=2, default=str))
        else:
            print(repr(val))
        print()

In [ ]:
# Show statsbomb_xg coverage
xg_col = [c for c in events.columns if 'statsbomb_xg' in c or (c.startswith('shot') and 'xg' in c.lower())]
print('xG columns:', xg_col)
if xg_col:
    col = xg_col[0]
    non_null = shots[col].notna().sum()
    print(f'{col}: {non_null}/{len(shots)} shots have xG ({100*non_null/len(shots):.1f}%)')
    print('Sample values:', shots[col].dropna().head(5).tolist())

## 3. frames_360.parquet

In [ ]:
frames = pd.read_parquet(CACHE / 'frames_360.parquet')
print('Shape:', frames.shape)
print()
print('dtypes:')
print(frames.dtypes.to_string())

In [ ]:
frames.head(3)

In [ ]:
# Show freeze_frame structure for one event
sample_id = frames['id'].iloc[0]
event_rows = frames[frames['id'] == sample_id]
print(f'event_uuid: {sample_id}')
print(f'visible_area: {event_rows["visible_area"].iloc[0]}')
print(f'Number of player entries in freeze_frame: {len(event_rows)}')
print()
print('Freeze frame player entries (all columns per row):')
print(event_rows[['teammate', 'actor', 'keeper', 'location']].to_string())

In [ ]:
# Verify join key: frames.id should match events.id
frame_ids = set(frames['id'].unique())
event_ids = set(events['id'].unique())
overlap = frame_ids & event_ids
print(f'Unique event IDs in frames_360: {len(frame_ids)}')
print(f'Unique event IDs in events:     {len(event_ids)}')
print(f'Overlap (join key works):       {len(overlap)}')
print(f'Coverage: {100*len(overlap)/len(frame_ids):.1f}% of frame events have a matching event row')

In [ ]:
# Pitch coordinate system verification
print('=== Pitch coordinate system ===')
locs = events['location'].dropna()
xs = locs.apply(lambda v: v[0] if isinstance(v, list) else None).dropna()
ys = locs.apply(lambda v: v[1] if isinstance(v, list) else None).dropna()
print(f'x range: {xs.min():.1f} – {xs.max():.1f}')
print(f'y range: {ys.min():.1f} – {ys.max():.1f}')

shot_locs = events[events['type'] == 'Shot']['location'].dropna()
shot_xs = shot_locs.apply(lambda v: v[0] if isinstance(v, list) else None).dropna()
print(f'\nShot x range (should cluster near 120): {shot_xs.min():.1f} – {shot_xs.max():.1f}')
print(f'Shot x median: {shot_xs.median():.1f}')

In [ ]:
# Sample JSON rows for SCHEMA.md
print('=== Sample matches row (JSON) ===')
print(matches.iloc[0].to_json(indent=2, default_handler=str))
print()
print('=== Sample Shot event row (JSON) ===')
shot_row = events[events['type']=='Shot'].iloc[0].dropna()
print(shot_row.to_json(indent=2, default_handler=str))

In [ ]:
print('=== Sample frames_360 event (JSON — one event_uuid) ===')
sample = frames[frames['id'] == sample_id].head(3)
print(sample.to_json(orient='records', indent=2, default_handler=str))